# Data Cleaning & ETL Pipeline — Amazon Sales
> **Improved version:** modular functions, DRY code, robust error handling, logging, and no duplicate cells.

## 0. Imports & Configuration

In [1]:
import logging
import urllib.parse

import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
AMAZON_PATH   = "../data/Amazon Sale Report.csv"
INTL_PATH     = "../data/International Sale Report.csv"
OUTPUT_PATH   = "../data/Cleaned_Master_Sales.csv"
UNKNOWN_TOKEN = "Unknown"   # single source of truth for fallback label

# ── Pandas display ───────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)

## 1. Load Data

In [2]:
def load_csv(path: str, **kwargs) -> pd.DataFrame:
    """Load a CSV and log its shape."""
    df = pd.read_csv(path, low_memory=False, **kwargs)
    logger.info("Loaded '%s' → shape %s", path, df.shape)
    return df


df_amazon = load_csv(AMAZON_PATH)
df_amazon.head()

2026-06-17 08:13:58 | INFO | Loaded '../data/Amazon Sale Report.csv' → shape (128975, 24)


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,ASIN,Courier Status,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


## 2. Data Quality Findings

| Issue | Column(s) | Action |
|---|---|---|
| Wrong dtype | `Date` (str) | → `datetime` |
| Wrong dtype | `ship-postal-code` (float) | → `str` |
| ~7 000 nulls | `Amount`, `currency` | → impute by status |
| >60 % nulls | `fulfilled-by` | → keep, document |
| Artifact column | `Unnamed: 22` | → drop |

In [3]:
# Quick diagnostic — run once, read results, do not repeat later
print("Shape:", df_amazon.shape)
df_amazon.info()

Shape: (128975, 24)
<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount              12118

## 3. Cleaning Pipeline

All transformations are wrapped in small, testable functions.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1  Drop artifact columns
# ─────────────────────────────────────────────────────────────────────────────
def drop_artifact_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Remove structurally empty / unnamed columns."""
    artifact_cols = [c for c in df.columns if c.startswith("Unnamed")]
    df = df.drop(columns=artifact_cols, errors="ignore")
    logger.info("Dropped artifact columns: %s", artifact_cols)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3.2  Date standardisation
# ─────────────────────────────────────────────────────────────────────────────
def parse_dates(df: pd.DataFrame, col: str = "Date") -> pd.DataFrame:
    """Coerce a string column to datetime; log the range."""
    df[col] = pd.to_datetime(df[col], errors="coerce", format="mixed")
    nat_count = df[col].isna().sum()
    logger.info(
        "'%s' parsed → range [%s, %s] | NaT introduced: %d",
        col, df[col].min(), df[col].max(), nat_count,
    )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3.3  Financial imputation
# ─────────────────────────────────────────────────────────────────────────────
def impute_financials(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fill missing Amount with 0.0 (cancelled & other anomalies).
    Fill missing currency with the mode.

    NOTE: Shipped orders with a missing Amount are filled with 0.0 as a
    conservative baseline. Flag them for business review before reporting.
    """
    # Log breakdown before imputing so analysts can audit
    missing_status = df.loc[df["Amount"].isna(), "Status"].value_counts()
    logger.info("Status breakdown for null Amount:\n%s", missing_status.to_string())

    df["Amount"] = df["Amount"].fillna(0.0)

    dominant_currency = df["currency"].mode()[0]
    df["currency"] = df["currency"].fillna(dominant_currency)
    logger.info("Imputed currency nulls with mode → '%s'", dominant_currency)

    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3.4  Geographic standardisation
# ─────────────────────────────────────────────────────────────────────────────
def standardise_geo(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise postal code dtype and title-case city/state."""
    # Postal code: float → clean string, preserve leading zeros
    df["ship-postal-code"] = (
        df["ship-postal-code"]
        .dropna()
        .astype(int)
        .astype(str)
        .reindex(df.index)          # re-align after dropna
    )
    # Title-case text geo columns
    for col in ("ship-city", "ship-state"):
        df[col] = df[col].str.strip().str.title()

    logger.info("Geographic columns standardised.")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3.5  Apply the full pipeline to df_amazon
# ─────────────────────────────────────────────────────────────────────────────
df_amazon = (
    df_amazon
    .pipe(drop_artifact_columns)
    .pipe(parse_dates)
    .pipe(impute_financials)
    .pipe(standardise_geo)
)

logger.info("Amazon dataset cleaned → shape %s", df_amazon.shape)

2026-06-17 08:13:59 | INFO | Dropped artifact columns: ['Unnamed: 22']
2026-06-17 08:13:59 | INFO | 'Date' parsed → range [2022-03-31 00:00:00, 2022-06-29 00:00:00] | NaT introduced: 0
2026-06-17 08:13:59 | INFO | Status breakdown for null Amount:
Status
Cancelled                       7566
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
2026-06-17 08:13:59 | INFO | Imputed currency nulls with mode → 'INR'
2026-06-17 08:13:59 | INFO | Geographic columns standardised.
2026-06-17 08:13:59 | INFO | Amazon dataset cleaned → shape (128975, 23)


## 4. Consolidation — Merge Local + International

In [5]:
df_intl = load_csv(INTL_PATH)

# Tag channels before concatenation
df_amazon["Market_Channel"] = "Local_Amazon"
df_intl["Market_Channel"]   = "International"

df_master = pd.concat([df_amazon, df_intl], axis=0, ignore_index=True)
logger.info("Master dataset shape: %s", df_master.shape)

# Export
df_master.to_csv(OUTPUT_PATH, index=False)
logger.info("Exported cleaned master → '%s'", OUTPUT_PATH)

2026-06-17 08:13:59 | INFO | Loaded '../data/International Sale Report.csv' → shape (37432, 10)
2026-06-17 08:13:59 | INFO | Master dataset shape: (166407, 30)
2026-06-17 08:14:03 | INFO | Exported cleaned master → '../data/Cleaned_Master_Sales.csv'


## 5. Database Connection

In [6]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# %pip install sqlalchemy pyodbc   # uncomment if needed

DB_CONFIG = {
    "driver":   "ODBC Driver 18 for SQL Server",
    "server":   "localhost",
    "database": "ECommerceSalesDB",
    "trusted":  True,            # Windows auth; set False + add user/pwd for SQL auth
}

connection_string = (
    f"DRIVER={{{DB_CONFIG['driver']}}};"
    f"SERVER={DB_CONFIG['server']};"
    f"DATABASE={DB_CONFIG['database']};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

# Test connection explicitly
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
logger.info("SQL Server connection verified.")

2026-06-17 08:14:04 | INFO | SQL Server connection verified.


## 6. Dimension Table Injection

A helper function replaces the repeated copy-paste pattern for each dimension.

In [7]:
def normalise_text_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Fill nulls with UNKNOWN_TOKEN and strip whitespace for a list of columns."""
    for col in cols:
        df[col] = df[col].fillna(UNKNOWN_TOKEN).astype(str).str.strip()
    return df


def load_dimension(
    df: pd.DataFrame,
    table_name: str,
    cols: list[str],
    rename_map: dict | None = None,
) -> None:
    """
    Extract a dimension slice from df, add a guaranteed 'Unknown' fallback row,
    deduplicate, and bulk-insert into SQL Server.
    """
    dim = df[cols].copy()
    if rename_map:
        dim.rename(columns=rename_map, inplace=True)

    clean_cols = list(dim.columns)
    dim = normalise_text_cols(dim, clean_cols)

    # Guaranteed fallback row
    fallback = pd.DataFrame([[UNKNOWN_TOKEN] * len(clean_cols)], columns=clean_cols)
    dim = pd.concat([fallback, dim]).drop_duplicates().reset_index(drop=True)

    dim.to_sql(table_name, con=engine, if_exists="append", index=False)
    logger.info("Loaded %d rows → %s", len(dim), table_name)


# ── Inject dimensions ─────────────────────────────────────────────────────────
load_dimension(
    df_master,
    "Dim_Status",
    ["Status", "Courier Status", "Fulfilment", "Market_Channel"],
    rename_map={"Courier Status": "Courier_Status"},
)

load_dimension(
    df_master,
    "Dim_Products",
    ["SKU", "ASIN", "Style", "Category", "Size"],
)

load_dimension(
    df_master,
    "Dim_Geography",
    ["ship-city", "ship-state", "ship-postal-code"],
    rename_map={
        "ship-city":        "Ship_City",
        "ship-state":       "Ship_State",
        "ship-postal-code": "Ship_Postal_Code",
    },
)

2026-06-17 08:14:04 | INFO | Loaded 24 rows → Dim_Status
2026-06-17 08:14:06 | INFO | Loaded 18726 rows → Dim_Products
2026-06-17 08:14:07 | INFO | Loaded 14470 rows → Dim_Geography


## 7. Fact Table Mapping & Bulk Insert

In [8]:
def fetch_dim(query: str) -> pd.DataFrame:
    """Read a dimension table and normalise all string columns."""
    df = pd.read_sql(query, con=engine)
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
    return df


df_status_sql   = fetch_dim("SELECT Status_ID, Status, Courier_Status, Fulfilment, Market_Channel FROM Dim_Status")
df_products_sql = fetch_dim("SELECT Product_ID, SKU, ASIN, Style, Category, Size FROM Dim_Products")
df_geo_sql      = fetch_dim("SELECT Location_ID, Ship_City, Ship_State, Ship_Postal_Code FROM Dim_Geography")

# Dynamic fallback IDs (avoids hard-coding)
fallback_status_id   = int(df_status_sql.loc[df_status_sql["Status"]    == UNKNOWN_TOKEN, "Status_ID"].iloc[0])
fallback_product_id  = int(df_products_sql.loc[df_products_sql["SKU"]   == UNKNOWN_TOKEN, "Product_ID"].iloc[0])
fallback_location_id = int(df_geo_sql.loc[df_geo_sql["Ship_City"]       == UNKNOWN_TOKEN, "Location_ID"].iloc[0])

logger.info("Fallback IDs → status=%d | product=%d | location=%d",
            fallback_status_id, fallback_product_id, fallback_location_id)

C:\Users\52444\AppData\Local\Temp\ipykernel_20572\3521035019.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\52444\AppData\Local\Temp\ipykernel_20572\3521035019.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

In [9]:
# ── Prepare fact mapping frame ────────────────────────────────────────────────
RENAME_FOR_FACT = {
    "Courier Status":   "Courier_Status",
    "ship-city":        "Ship_City",
    "ship-state":       "Ship_State",
    "ship-postal-code": "Ship_Postal_Code",
}
TEXT_COLS_FACT = [
    "Status", "Courier_Status", "Fulfilment", "Market_Channel",
    "SKU", "ASIN", "Style", "Category", "Size",
    "Ship_City", "Ship_State", "Ship_Postal_Code",
]

df_fact = (
    df_master
    .rename(columns=RENAME_FOR_FACT)
    .pipe(normalise_text_cols, TEXT_COLS_FACT)
)

# Surrogate key joins
df_fact = (
    df_fact
    .merge(df_status_sql,   on=["Status", "Courier_Status", "Fulfilment", "Market_Channel"], how="left")
    .merge(df_products_sql, on=["SKU", "ASIN", "Style", "Category", "Size"],                how="left")
    .merge(df_geo_sql,      on=["Ship_City", "Ship_State", "Ship_Postal_Code"],              how="left")
)

# Fill unmapped records with fallback IDs
df_fact["Status_ID"]   = df_fact["Status_ID"].fillna(fallback_status_id).astype(int)
df_fact["Product_ID"]  = df_fact["Product_ID"].fillna(fallback_product_id).astype(int)
df_fact["Location_ID"] = df_fact["Location_ID"].fillna(fallback_location_id).astype(int)

# Keep only fact schema columns
FACT_COLS = ["Order ID", "Date", "Qty", "Amount", "Status_ID", "Product_ID", "Location_ID"]
df_fact = (
    df_fact[FACT_COLS]
    .rename(columns={"Order ID": "Order_ID"})
    .dropna(subset=["Order_ID"])
)
# Guard against string 'nan'
df_fact = df_fact[df_fact["Order_ID"].astype(str).str.lower() != "nan"]

logger.info("Fact frame ready → %d rows", len(df_fact))

# Bulk insert
df_fact.to_sql("Fact_Sales", con=engine, if_exists="append", index=False, chunksize=10_000)
logger.info("Fact_Sales loaded successfully.")

2026-06-17 08:14:10 | INFO | Fact frame ready → 1031792 rows
2026-06-17 08:15:20 | INFO | Fact_Sales loaded successfully.


## 8. QA & Reconciliation

In [10]:
def qa_reconcile(df_source: pd.DataFrame) -> None:
    """Compare row counts and revenue totals between Pandas source and SQL destination."""
    # Source counts (mirror pipeline exclusion logic)
    df_clean = df_source.dropna(subset=["Order ID"])
    df_clean = df_clean[df_clean["Order ID"].astype(str).str.lower() != "nan"]
    src_rows    = len(df_clean)
    src_revenue = df_clean["Amount"].sum()

    # Destination counts
    dst_rows    = pd.read_sql("SELECT COUNT(*) AS n FROM Fact_Sales", con=engine).iloc[0]["n"]
    dst_revenue = pd.read_sql("SELECT SUM(Amount) AS s FROM Fact_Sales", con=engine).iloc[0]["s"]

    row_match = src_rows == dst_rows
    rev_match = abs(src_revenue - dst_revenue) < 0.01

    print(f"{'':─<55}")
    print(f" Rows    → Source: {src_rows:>10,} | DB: {dst_rows:>10,} | {'✓ PASS' if row_match else '✗ FAIL'}")
    print(f" Revenue → Source: ${src_revenue:>12,.2f} | DB: ${dst_revenue:>12,.2f} | {'✓ PASS' if rev_match else '✗ FAIL'}")
    print(f"{'':─<55}")

    if not row_match:
        logger.warning("Row count mismatch: source=%d db=%d", src_rows, dst_rows)
    if not rev_match:
        logger.warning("Revenue mismatch: diff=%.2f", abs(src_revenue - dst_revenue))


qa_reconcile(df_master)

2026-06-17 08:15:21 | WARNING | Row count mismatch: source=128975 db=1289742
2026-06-17 08:15:21 | WARNING | Revenue mismatch: diff=707326092.70


───────────────────────────────────────────────────────
 Rows    → Source:    128,975 | DB:  1,289,742 | ✗ FAIL
 Revenue → Source: $78,592,678.30 | DB: $785,918,771.00 | ✗ FAIL
───────────────────────────────────────────────────────
